# 01 - NanoVNA acquisition and rfmeasurement records

Acquire SOL repetitions, save them as Touchstone files, and validate each sweep with `rfmeasurement`.

In [ ]:
from pathlib import Path
import numpy as np
import skrf as rf
import pynanovna
from pynanovna.hardware import Hardware as hw
from rfmeasurement.domain import Measurement, MeasurementContext
from rfmeasurement.validation import validate

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
PORT = None  # Ejemplo: '/dev/cu.usbmodem00000000001A1'
F_START_HZ = 50e3
F_STOP_HZ = 1.45e9
N_POINTS = 101
N_REPETITIONS = 10
OUTPUT_ROOT = ROOT / 'data' / 'calibration'

def wrap_measurement(network, identifier):
    return Measurement(
        data=network,
        context=MeasurementContext(
            dut=identifier, instrument='NanoVNA',
            calibration='uncalibrated/raw',
            frequency_range_hz=(float(network.frequency.start), float(network.frequency.stop)),
            confidence={'instrument': 'specified', 'frequency_range_hz': 'measured'},
        ),
    )

def connect_vna(port=None, vna_index=0):
    interfaces = hw.get_interfaces()
    if not interfaces:
        raise RuntimeError('No se detecto ningun NanoVNA. Comprueba USB y alimentacion.')
    if port is not None:
        matches = [index for index, interface in enumerate(interfaces) if interface.port == port]
        if not matches:
            available = [interface.port for interface in interfaces]
            raise ValueError(f'Puerto no encontrado: {port}. Disponibles: {available}')
        vna_index = matches[0]
    if not 0 <= vna_index < len(interfaces):
        raise IndexError(f'vna_index fuera de rango: {vna_index}')

    vna = pynanovna.VNA(vna_index=vna_index, logging_level='info')
    if not getattr(vna, 'connected', False) or getattr(vna, 'vna', None) is None:
        interface = getattr(vna, 'iface', interfaces[vna_index])
        try:
            interface.close()
        except Exception:
            pass
        raise RuntimeError(
            'pynanovna encontro el puerto pero no pudo inicializar el backend NanoVNA. '
            'Cierra otros programas que usen el puerto y vuelve a intentarlo.'
        )
    return vna

def close_vna(vna):
    interface = getattr(vna, 'iface', None)
    if interface is not None:
        try:
            interface.close()
        except Exception:
            pass

def acquire_sweep(vna):
    vna.set_sweep(int(F_START_HZ), int(F_STOP_HZ), N_POINTS)
    s11, s21, frequencies = vna.sweep()
    frequencies = np.asarray(frequencies, dtype=float)
    s11, s21 = np.asarray(s11), np.asarray(s21)
    matrix = np.zeros((len(frequencies), 2, 2), dtype=complex)
    matrix[:, 0, 0] = s11
    matrix[:, 1, 0] = s21
    matrix[:, 0, 1] = s21
    frequency = rf.Frequency.from_f(frequencies, unit='hz')
    return rf.Network(frequency=frequency, s=matrix, z0=[50, 50])

In [5]:
STANDARD = 'open'  # Ejecuta despues con 'short' y 'load'
target = OUTPUT_ROOT / STANDARD
target.mkdir(parents=True, exist_ok=True)
vna = connect_vna(PORT)
try:
    for repetition in range(1, N_REPETITIONS + 1):
        network = acquire_sweep(vna)
        identifier = f'{STANDARD}_test_{repetition:02d}'
        measurement = wrap_measurement(network, identifier)
        report = validate(measurement)
        if report.has_failures:
            raise RuntimeError(f'Validacion fallida en {identifier}')
        network.write_touchstone(str(target / f'test_{repetition}'))
        print(identifier, 'OK')
finally:
    close_vna(vna)

RuntimeError: No se detecto ningun NanoVNA. Comprueba USB y alimentacion.